In [60]:
import os
import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from collections import Counter
from imblearn.over_sampling import RandomOverSampler

from xgboost import XGBClassifier

import joblib

In [61]:
DATASET_PATH = r"../../Dataset/ADFA_dataset/ADFA-IDS_DATASETS/ADFA-LD"

In [62]:
print(os.path.exists(DATASET_PATH))

True


In [63]:
all_data = []

all_labels = []

In [64]:
# =========================================
# READ ADFA-LD DATASET
# =========================================

import re

all_data = []

all_labels = []

print("Reading ADFA-LD Dataset...\n")

for root, dirs, files in os.walk(DATASET_PATH):

    for file in tqdm(files):

        # =========================================
        # ONLY READ TXT FILES
        # =========================================

        if file.endswith(".txt"):

            file_path = os.path.join(root, file)

            try:

                with open(file_path, "r") as f:

                    # =========================================
                    # READ SYSCALL SEQUENCE
                    # =========================================

                    content = f.read().strip()

                    syscalls = content.split()

                    syscall_sequence = [

                        int(x)

                        for x in syscalls

                        if x.isdigit()
                    ]

                    # =========================================
                    # SKIP EMPTY FILES
                    # =========================================

                    if len(syscall_sequence) > 0:

                        all_data.append(syscall_sequence)

                        # =========================================
                        # EXTRACT FOLDER NAME
                        # =========================================

                        folder_name = os.path.basename(root)

                        # =========================================
                        # NORMAL DATA
                        # =========================================

                        if folder_name in [

                            "Training_Data_Master",
                            "Validation_Data_Master"
                        ]:

                            label = "Normal"

                        # =========================================
                        # ATTACK DATA
                        # =========================================

                        else:

                            # Remove _1, _2, _3 etc.
                            label = re.sub(
                                r'_\d+$',
                                '',
                                folder_name
                            )

                        all_labels.append(label)

            except Exception as e:

                print("\nError reading file:")

                print(file_path)

                print("Exception:", e)

print("\n===================================")
print("Dataset Loading Complete")
print("===================================")

print(f"Total Samples: {len(all_data)}")

print(f"Total Labels: {len(all_labels)}")

Reading ADFA-LD Dataset...



0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
100%|██████████| 4372/4372 [00:01<00:00, 3303.65it/s]


Dataset Loading Complete
Total Samples: 5951
Total Labels: 5951


In [65]:
# =========================================
# CHECK UNIQUE LABELS
# =========================================

print("\nUnique Labels Found:\n")

print(set(all_labels))


Unique Labels Found:

{'Normal', 'Meterpreter', 'Web_Shell', 'Adduser', 'Hydra_SSH', 'Java_Meterpreter', 'Hydra_FTP'}


In [66]:
#Dataset Statistics 
print("Total samples:", len(all_data))

print("Total labels:", len(all_labels))

Total samples: 5951
Total labels: 5951


In [67]:
# =========================================
# LABEL DISTRIBUTION
# =========================================

unique, counts = np.unique(
    all_labels,
    return_counts=True
)

print("\nLabel Distribution:\n")

for u, c in zip(unique, counts):

    print(f"{u}: {c}")


Label Distribution:

Adduser: 91
Hydra_FTP: 162
Hydra_SSH: 176
Java_Meterpreter: 124
Meterpreter: 75
Normal: 5205
Web_Shell: 118


In [68]:
#Sequence Length 
MAX_LEN = 500

In [69]:
#Padding Sequences
processed_data = []

for seq in all_data:

    if len(seq) >= MAX_LEN:

        seq = seq[:MAX_LEN]

    else:

        seq = seq + [0] * (MAX_LEN - len(seq))

    processed_data.append(seq)

In [70]:
#Convert to numpy arrays
X = np.array(processed_data)

y = np.array(all_labels)

print(X.shape)

print(y.shape)

(5951, 500)
(5951,)


In [71]:
print("\nFeature Vector Length:", X.shape[1])


Feature Vector Length: 500


In [72]:
#Encode Labels
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

In [73]:
#Split Data
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_encoded,

    test_size=0.2,

    random_state=42,

    stratify=y_encoded
)

In [74]:

print("\nClass Distribution Before Balancing:")

print(Counter(y_train))


Class Distribution Before Balancing:
Counter({np.int64(5): 4163, np.int64(2): 141, np.int64(1): 130, np.int64(3): 99, np.int64(6): 94, np.int64(0): 73, np.int64(4): 60})


In [75]:
# =========================================
# BALANCE TRAINING DATA
# =========================================

ros = RandomOverSampler(
    random_state=42
)

X_train, y_train = ros.fit_resample(
    X_train,
    y_train
)

print("\nBalanced Training Distribution:")

print(Counter(y_train))


Balanced Training Distribution:
Counter({np.int64(5): 4163, np.int64(3): 4163, np.int64(6): 4163, np.int64(0): 4163, np.int64(2): 4163, np.int64(4): 4163, np.int64(1): 4163})


In [76]:
print("\nTraining Samples:", len(X_train))

print("Testing Samples:", len(X_test))


Training Samples: 29141
Testing Samples: 1191


In [77]:
model = XGBClassifier(

    n_estimators=150,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    objective='multi:softprob',

    num_class=len(np.unique(y_encoded)),

    eval_metric='mlogloss',

    n_jobs=-1,

    random_state=42
)

In [78]:
model.fit(

    X_train,
    y_train,

    eval_set=[(X_test, y_test)],

    verbose=True
)

[0]	validation_0-mlogloss:1.89435
[1]	validation_0-mlogloss:1.85015
[2]	validation_0-mlogloss:1.80063
[3]	validation_0-mlogloss:1.75090
[4]	validation_0-mlogloss:1.71169
[5]	validation_0-mlogloss:1.67578
[6]	validation_0-mlogloss:1.64379
[7]	validation_0-mlogloss:1.61385
[8]	validation_0-mlogloss:1.58086
[9]	validation_0-mlogloss:1.55417
[10]	validation_0-mlogloss:1.53003
[11]	validation_0-mlogloss:1.49889
[12]	validation_0-mlogloss:1.46915
[13]	validation_0-mlogloss:1.43745
[14]	validation_0-mlogloss:1.41058
[15]	validation_0-mlogloss:1.38728
[16]	validation_0-mlogloss:1.36552
[17]	validation_0-mlogloss:1.34432
[18]	validation_0-mlogloss:1.32297
[19]	validation_0-mlogloss:1.30307
[20]	validation_0-mlogloss:1.28246
[21]	validation_0-mlogloss:1.26375
[22]	validation_0-mlogloss:1.24259
[23]	validation_0-mlogloss:1.22211
[24]	validation_0-mlogloss:1.20657
[25]	validation_0-mlogloss:1.19146
[26]	validation_0-mlogloss:1.17663
[27]	validation_0-mlogloss:1.16070
[28]	validation_0-mlogloss:1.1

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [79]:
print(model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=-1, num_class=7, ...)


In [80]:
#Prediction 
y_prob = model.predict_proba(X_test)

y_pred = np.argmax(y_prob, axis=1)

In [81]:
# =========================================
# TRAINING ACCURACY
# =========================================

train_pred = model.predict(X_train)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

print(f"\nTraining Accuracy: {train_accuracy:.4f}")


Training Accuracy: 0.9691


In [82]:
#Accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8346


In [83]:
#Classification Report
print(

    classification_report(

        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.25      0.33      0.29        18
           1       0.15      0.19      0.16        32
           2       0.30      0.51      0.38        35
           3       0.33      0.48      0.39        25
           4       0.23      0.20      0.21        15
           5       0.96      0.90      0.93      1042
           6       0.26      0.42      0.32        24

    accuracy                           0.83      1191
   macro avg       0.35      0.43      0.38      1191
weighted avg       0.87      0.83      0.85      1191



In [84]:
#Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[  6   2   1   1   1   6   1]
 [  2   6   6   5   3   9   1]
 [  1   2  18   2   2   9   1]
 [  2   0   3  12   0   5   3]
 [  2   3   0   2   3   4   1]
 [  7  26  31  13   4 939  22]
 [  4   2   1   1   0   6  10]]


In [85]:
# =========================================
# PREDICTION DISTRIBUTION
# =========================================

unique, counts = np.unique(
    y_pred,
    return_counts=True
)

print("\nPrediction Distribution:\n")

for u, c in zip(unique, counts):

    label = label_encoder.inverse_transform([u])[0]

    print(f"{label}: {c}")


Prediction Distribution:

Adduser: 24
Hydra_FTP: 41
Hydra_SSH: 60
Java_Meterpreter: 36
Meterpreter: 13
Normal: 978
Web_Shell: 39


In [86]:
os.makedirs(

    "../../trained_models/linux_ids",

    exist_ok=True
)

In [87]:
joblib.dump(

    model,

    "../../trained_models/linux_ids/linux_xgboost_model.pkl"
)

['../../trained_models/linux_ids/linux_xgboost_model.pkl']

In [88]:
joblib.dump(

    label_encoder,

    "../../trained_models/linux_ids/linux_label_encoder.pkl"
)

['../../trained_models/linux_ids/linux_label_encoder.pkl']

In [89]:
print("===================================")

print("AEGIS Linux IDS Training Complete")

print("===================================")

print("Linux IDS Model Saved Successfully")

AEGIS Linux IDS Training Complete
Linux IDS Model Saved Successfully
